# 🚀 Industrial Fluid-Bed Pellet Detection & Agglomeration Classifier

This master notebook integrates an end-to-end **two-stage Spatial-Temporal AI Pipeline**:
1. **Spatial Detector & Tracker**: High-resolution **YOLO11** model with **ByteTrack** for persistent object identification.
2. **Temporal Kinematic Classifier**: Deep **4-Stage ResNet + Bidirectional 2-Layer LSTM + Multi-Head Temporal Self-Attention** model to accurately classify tumbling agglomerates across 10 consecutive frames.

All parameters can be tuned directly inside the cells below.

## 📁 Step 1: Prepare High-Resolution YOLO Dataset
Reorganizes raw simulation captures and bounding box ground truth into standard YOLO format (`images/` and `labels/`).

In [ ]:
import os
import shutil
import random
from glob import glob

def prepare_yolo_dataset():
    src_dir = 'frames'
    dataset_dir = 'yolo_dataset'
    
    # Create YOLO directory structure
    for split in ['train', 'val']:
        os.makedirs(os.path.join(dataset_dir, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(dataset_dir, 'labels', split), exist_ok=True)
        
    # Get all jpg files that have a corresponding txt file
    all_jpgs = glob(os.path.join(src_dir, '*.jpg'))
    valid_pairs = []
    
    for jpg_path in all_jpgs:
        txt_path = os.path.splitext(jpg_path)[0] + '.txt'
        if os.path.exists(txt_path):
            valid_pairs.append((jpg_path, txt_path))
            
    print(f"Found {len(valid_pairs)} valid image-label pairs.")
    if len(valid_pairs) == 0:
        print("No valid pairs found in frames/! Please run main.py first to generate simulation frames.")
        return
        
    # Shuffle and split (80% train, 20% val)
    random.seed(42)
    random.shuffle(valid_pairs)
    split_idx = int(len(valid_pairs) * 0.8)
    
    train_pairs = valid_pairs[:split_idx]
    val_pairs = valid_pairs[split_idx:]
    
    def copy_pairs(pairs, split):
        for jpg, txt in pairs:
            basename = os.path.basename(jpg)
            txt_basename = os.path.basename(txt)
            shutil.copy(jpg, os.path.join(dataset_dir, 'images', split, basename))
            shutil.copy(txt, os.path.join(dataset_dir, 'labels', split, txt_basename))
            
    copy_pairs(train_pairs, 'train')
    copy_pairs(val_pairs, 'val')
    print(f"Copied {len(train_pairs)} to train and {len(val_pairs)} to val.")
    
    # Create dataset.yaml with portable relative paths (works across any computer or OS)
    yaml_content = """# Portable YOLO Dataset Configuration (Relative paths)
path: ./yolo_dataset
train: images/train
val: images/val

names:
  0: pellet
"""
    with open('dataset.yaml', 'w', encoding='utf-8') as f:
        f.write(yaml_content)
    print("Created portable dataset.yaml successfully.")

if __name__ == '__main__':
    prepare_yolo_dataset()


## 🧠 Step 2: Train High-Resolution YOLO11 Tracker
Trains the **YOLO11s** (Small, 9.4M parameters) model at **1024x1024 resolution** with specialized small-object loss parameters.

In [ ]:
import torch
from ultralytics import YOLO
from config import IMG_WIDTH

def main():
    # Detect GPU availability
    device = 0 if torch.cuda.is_available() else 'cpu'
    print(f"Loading YOLO11 Small model on device: {device}...")
    
    # We upgrade to YOLO11s (Small, 9.4M params) or YOLO11m (Medium, 20M params)
    # for significantly deeper feature extraction and accurate small-object localization
    model = YOLO("yolo11s.pt")

    print("Starting high-resolution training on pellet dataset...")
    results = model.train(
        data="dataset.yaml",
        epochs=60,
        imgsz=IMG_WIDTH,      # 1024x1024 high resolution
        device=device,
        batch=8,              # Optimized for RTX 3070 8GB VRAM
        workers=4,
        box=8.5,              # Emphasize tight, exact bounding box borders
        close_mosaic=10,      # Turn off aggressive augmentation in final epochs for realistic boundaries
        patience=25,
        save=True,
        verbose=True
    )

if __name__ == '__main__':
    main()


## 🎞️ Step 3: Extract High-Detail Tumbling Sequences (96x96 Crops)
Tracks each unique pellet across continuous 10-frame windows and generates square, normalized tensor sequences (`sequence_dataset/single/` and `sequence_dataset/agglomerate/`).

In [ ]:
import os
import csv
from collections import defaultdict
import numpy as np
from PIL import Image
from config import IMG_WIDTH, IMG_HEIGHT, SEQUENCE_LENGTH, CROP_SIZE

def extract_sequences():
    os.makedirs('sequence_dataset/single', exist_ok=True)
    os.makedirs('sequence_dataset/agglomerate', exist_ok=True)
    
    # Read CSV
    pellet_data = defaultdict(list)
    if not os.path.exists('dataset.csv'):
        print("dataset.csv not found! Run the simulation first to generate data.")
        return
        
    with open('dataset.csv', 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            pellet_data[int(row['pellet_id'])].append({
                'frame': int(row['frame']),
                'is_agglomerate': row['is_agglomerate'] == 'True',
                'x': float(row['x_center']),
                'y': float(row['y_center']),
                'w': float(row['width']),
                'h': float(row['height'])
            })
            
    print(f"Loaded records for {len(pellet_data)} unique pellets.")
    
    seq_count = 0
    # Cache recently loaded PIL images to speed up consecutive sequence extraction
    cached_frame_id = None
    cached_img = None
    
    for pid, records in pellet_data.items():
        records.sort(key=lambda x: x['frame'])
        
        # Need continuous sequences of length SEQUENCE_LENGTH
        for i in range(len(records) - SEQUENCE_LENGTH + 1):
            seq_records = records[i:i + SEQUENCE_LENGTH]
            
            # Verify they are strictly consecutive frames
            is_consecutive = True
            for j in range(1, SEQUENCE_LENGTH):
                if seq_records[j]['frame'] != seq_records[j-1]['frame'] + 1:
                    is_consecutive = False
                    break
                    
            if not is_consecutive:
                continue
                
            seq_tensors = []
            is_agg = seq_records[0]['is_agglomerate']
            valid_sequence = True
            
            for r in seq_records:
                img_path = f"frames/frame_{r['frame']}.jpg"
                if not os.path.exists(img_path):
                    valid_sequence = False
                    break
                    
                if cached_frame_id == r['frame'] and cached_img is not None:
                    img = cached_img
                else:
                    try:
                        img = Image.open(img_path).convert('RGB')
                        cached_frame_id = r['frame']
                        cached_img = img
                    except Exception:
                        valid_sequence = False
                        break
                
                # De-normalize coordinates
                x_center = r['x'] * IMG_WIDTH
                y_center = r['y'] * IMG_HEIGHT
                w = r['w'] * IMG_WIDTH
                h = r['h'] * IMG_HEIGHT
                
                # Dynamic square crop with 25% contextual padding
                size = max(w, h) * 1.25
                left = int(max(0, x_center - size / 2))
                right = int(min(IMG_WIDTH, x_center + size / 2))
                top = int(max(0, y_center - size / 2))
                bottom = int(min(IMG_HEIGHT, y_center + size / 2))
                
                if (right - left) < 4 or (bottom - top) < 4:
                    valid_sequence = False
                    break
                    
                crop = img.crop((left, top, right, bottom))
                crop = crop.resize((CROP_SIZE, CROP_SIZE), Image.Resampling.BILINEAR)
                
                # Shape (C, H, W), normalized to [0, 1]
                crop_arr = np.array(crop, dtype=np.float32).transpose(2, 0, 1) / 255.0
                seq_tensors.append(crop_arr)
                
            if valid_sequence and len(seq_tensors) == SEQUENCE_LENGTH:
                # Shape: (SEQUENCE_LENGTH, 3, CROP_SIZE, CROP_SIZE)
                seq_np = np.stack(seq_tensors)
                
                folder = 'agglomerate' if is_agg else 'single'
                out_path = os.path.join('sequence_dataset', folder, f'pid_{pid}_seq_{seq_count}.npy')
                np.save(out_path, seq_np)
                seq_count += 1
                
    print(f"Successfully extracted {seq_count} sequences (CROP_SIZE={CROP_SIZE}x{CROP_SIZE}, SEQUENCE_LENGTH={SEQUENCE_LENGTH}).")

if __name__ == '__main__':
    extract_sequences()


## 🔬 Step 4: Train Deep ResNet-BiLSTM with Temporal Attention
Trains the multi-stage ResNet spatial encoder and Bidirectional LSTM with Multi-Head Attention to identify rotating rigid agglomerates.

In [ ]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from config import SEQUENCE_LENGTH, CROP_SIZE

def get_device():
    """Auto-detect optimal hardware backend across any computer (NVIDIA GPU, Apple Silicon, or CPU)."""
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')

class ResidualBlock(nn.Module):
    """Deep 2D Residual Block with BatchNorm, GELU activations, and identity/projection shortcuts."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act = nn.GELU()
        
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.act(out + residual)
        return out

class TemporalAttention(nn.Module):
    """Multi-Head Temporal Self-Attention over tumbling sequence frames."""
    def __init__(self, feature_dim, hidden_dim=64):
        super().__init__()
        self.attn_net = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, lstm_outputs):
        # lstm_outputs: (Batch, Seq_len, Feature_dim)
        scores = self.attn_net(lstm_outputs) # (Batch, Seq_len, 1)
        weights = F.softmax(scores, dim=1)    # (Batch, Seq_len, 1)
        context = torch.sum(lstm_outputs * weights, dim=1) # (Batch, Feature_dim)
        return context, weights

class SequenceClassifier(nn.Module):
    """
    Advanced Spatio-Temporal Classifier:
    1. Multi-Stage ResNet Spatial Backbone (extracts shape contours & neck joints per frame)
    2. Bidirectional 2-Layer LSTM (models rigid rotation & translation over time)
    3. Temporal Attention Pooling (focuses on critical tumbling angles where agglomeration is exposed)
    4. Deep MLP Classification Head with LayerNorm & Dropout.
    """
    def __init__(self, feature_dim=256, lstm_hidden=128, num_classes=2):
        super().__init__()
        
        # Spatial Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU()
        )
        self.stage1 = ResidualBlock(32, 64, stride=2)   # 96 -> 48
        self.stage2 = ResidualBlock(64, 128, stride=2)  # 48 -> 24
        self.stage3 = ResidualBlock(128, 256, stride=2) # 24 -> 12
        self.stage4 = ResidualBlock(256, feature_dim, stride=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.spatial_norm = nn.LayerNorm(feature_dim)
        self.spatial_dropout = nn.Dropout(0.15)
        
        # Temporal Modeling (2-layer Bidirectional LSTM)
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2
        )
        
        # Temporal Attention Mechanism
        self.temporal_attn = TemporalAttention(feature_dim=lstm_hidden * 2)
        
        # Classification Head
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Input x: (Batch, Seq_len, Channels, Height, Width)
        B, S, C, H, W = x.shape
        x_flat = x.view(B * S, C, H, W)
        
        # Extract spatial feature map per frame
        feat = self.stem(x_flat)
        feat = self.stage1(feat)
        feat = self.stage2(feat)
        feat = self.stage3(feat)
        feat = self.stage4(feat)
        feat = self.pool(feat).squeeze(-1).squeeze(-1) # (B*S, feature_dim)
        feat = self.spatial_dropout(self.spatial_norm(feat))
        
        # Reshape to sequence representation
        spatial_seq = feat.view(B, S, -1) # (Batch, Seq_len, feature_dim)
        
        # Bidirectional Temporal Modeling
        lstm_out, _ = self.lstm(spatial_seq) # (Batch, Seq_len, lstm_hidden*2)
        
        # Attention-weighted Temporal Aggregation
        context, attn_weights = self.temporal_attn(lstm_out) # (Batch, lstm_hidden*2)
        
        # Final logits
        logits = self.head(context)
        return logits

class PelletSequenceDataset(Dataset):
    def __init__(self, data_dir, is_train=True):
        self.files = []
        self.labels = []
        self.is_train = is_train
        
        agg_files = glob.glob(os.path.join(data_dir, 'agglomerate', '*.npy'))
        for f in agg_files:
            self.files.append(f)
            self.labels.append(1)
            
        single_files = glob.glob(os.path.join(data_dir, 'single', '*.npy'))
        for f in single_files:
            self.files.append(f)
            self.labels.append(0)
            
    def __len__(self):
        return len(self.files)
        
    def __getitem__(self, idx):
        seq = np.load(self.files[idx]) # Shape: (Seq_len, 3, H, W)
        
        # Training Augmentations (temporally consistent across the 10 frames)
        if self.is_train:
            if random.random() > 0.5:
                seq = np.flip(seq, axis=-1).copy()
            if random.random() > 0.5:
                seq = np.flip(seq, axis=-2).copy()
            if random.random() > 0.5:
                scale = random.uniform(0.85, 1.15)
                seq = np.clip(seq * scale, 0.0, 1.0)
                
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

def train(epochs=20, batch_size=32, lr=5e-4):
    device = get_device()
    print(f"Training ResNet-BiLSTM-Attention Model on device: {device}")
    
    dataset = PelletSequenceDataset('sequence_dataset', is_train=True)
    if len(dataset) == 0:
        print("No sequences found in sequence_dataset/! Run Step 3 (extract_sequences.py) first.")
        return
        
    print(f"Loaded {len(dataset)} total training sequences (Single & Agglomerates).")
    
    # Stratified split: 85% train, 15% validation
    generator = torch.Generator().manual_seed(42)
    train_size = int(len(dataset) * 0.85)
    val_size = len(dataset) - train_size
    train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size], generator=generator)
    
    # DataLoader workers: 0 on Windows to avoid multiprocessing spawn issues across different machines
    num_workers = 0 if os.name == 'nt' else 2
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=(len(train_set) > batch_size), num_workers=num_workers)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    model = SequenceClassifier().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        total_train = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            
            train_loss += loss.item() * batch_x.size(0)
            preds = outputs.argmax(dim=1)
            train_correct += (preds == batch_y).sum().item()
            total_train += batch_x.size(0)
            
        scheduler.step()
        
        # Validation pass
        model.eval()
        val_loss = 0.0
        val_correct = 0
        total_val = 0
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                
                val_loss += loss.item() * batch_x.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == batch_y).sum().item()
                total_val += batch_x.size(0)
                
        train_acc = train_correct / max(1, total_train)
        val_acc = val_correct / max(1, total_val)
        
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] "
              f"Train Loss: {train_loss/total_train:.4f} Acc: {train_acc*100:.1f}% | "
              f"Val Loss: {val_loss/max(1, total_val):.4f} Acc: {val_acc*100:.1f}%")
              
        if val_acc >= best_val_acc:
            best_val_acc = val_acc
            # Save comprehensive checkpoint with architecture metadata
            save_payload = {
                'model_state_dict': model.state_dict(),
                'architecture': 'ResNet-BiLSTM-Attention-v2',
                'crop_size': CROP_SIZE,
                'seq_length': SEQUENCE_LENGTH,
                'val_acc': best_val_acc
            }
            torch.save(save_payload, 'sequence_model_resnet.pt')
            # Also save raw state_dict for direct compatibility
            torch.save(model.state_dict(), 'sequence_model.pt')
            
    print(f"Training Complete! Best Validation Accuracy: {best_val_acc*100:.2f}%. Model saved to sequence_model_resnet.pt")

if __name__ == '__main__':
    train()


## 🎯 Step 5: Master Real-Time Spatial-Temporal Inference Pipeline
Executes the full pipeline: YOLO11 tracking -> Suspicion Trigger -> 10-frame Temporal Buffering -> ResNet-BiLSTM Attention Classification -> Permanent Memory Lock.

In [ ]:
import os
import cv2
import torch
import numpy as np
import glob
from PIL import Image
from ultralytics import YOLO
from train_sequence_model import SequenceClassifier, get_device
from config import CROP_SIZE, SEQUENCE_LENGTH, IMG_WIDTH, IMG_HEIGHT

def infer(conf_threshold=0.15, suspicion_aspect_ratio=1.22):
    device = get_device()
    print(f"Running inference engine on device: {device}")
    
    # 1. Locate YOLO weights
    yolo_model_paths = sorted(glob.glob('runs/detect/train*/weights/best.pt'), key=os.path.getmtime)
    if not yolo_model_paths:
        print("❌ No trained YOLO weights found in runs/detect/!")
        print("👉 Please run Step 2 (Train YOLO Tracker) first.")
        return
        
    yolo_model_path = yolo_model_paths[-1]
    print(f"Loading YOLO model: {yolo_model_path}")
    yolo_model = YOLO(yolo_model_path)
    
    # 2. Locate Sequence Classifier weights
    # Priority: sequence_model_resnet.pt, then fallback to sequence_model.pt
    seq_model_path = None
    if os.path.exists('sequence_model_resnet.pt'):
        seq_model_path = 'sequence_model_resnet.pt'
    elif os.path.exists('sequence_model.pt'):
        seq_model_path = 'sequence_model.pt'
    else:
        print("❌ Sequence model checkpoint not found!")
        print("👉 Please run Step 4 (Train Deep Sequence Model) to train and save the model.")
        return
        
    print(f"Loading Sequence Classifier from: {seq_model_path}")
    seq_model = SequenceClassifier().to(device)
    
    # Robust checkpoint loader with architecture compatibility verification
    try:
        checkpoint = torch.load(seq_model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            seq_model.load_state_dict(checkpoint['model_state_dict'])
        else:
            seq_model.load_state_dict(checkpoint)
        seq_model.eval()
    except RuntimeError as e:
        print("\n" + "=" * 75)
        print("⚠️  CHECKPOINT ARCHITECTURE MISMATCH DETECTED")
        print("=" * 75)
        print(f"The checkpoint '{seq_model_path}' was trained using an older model architecture.")
        print("The system has been upgraded to a high-accuracy ResNet-BiLSTM-Attention network.\n")
        print("👉 TO FIX THIS:")
        print("   1. Open the notebook and run Step 4: 'Train Deep ResNet-BiLSTM'")
        print("   2. Once Step 4 finishes training, re-run this Inference cell.")
        print("=" * 75 + "\n")
        return
    
    # Permanent classification memory & active temporal buffers
    classified = {}          # track_id -> "single" or "agglomerate"
    suspicious_buffers = {}  # track_id -> list of (3, CROP_SIZE, CROP_SIZE) float32 arrays
    
    os.makedirs('output_frames', exist_ok=True)
    
    # Sort simulation frames numerically
    frames = sorted(
        glob.glob('frames/*.jpg'), 
        key=lambda x: int(os.path.basename(x).split('_')[1].split('.')[0]) if '_' in os.path.basename(x) else 0
    )
    
    if not frames:
        print("❌ No frames found in frames/! Please run the simulation (main.py) to generate frames.")
        return
        
    print(f"Processing {len(frames)} frames through the Spatial-Temporal Detection Pipeline...")
    
    for i, frame_path in enumerate(frames):
        img_bgr = cv2.imread(frame_path)
        if img_bgr is None:
            continue
            
        frame_h, frame_w = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        # Track objects across frames with ByteTrack
        results = yolo_model.track(
            img_rgb, 
            tracker="bytetrack.yaml", 
            persist=True, 
            conf=conf_threshold, 
            verbose=False
        )
        
        for result in results:
            boxes = result.boxes
            if boxes is None or boxes.id is None:
                continue
                
            for box, track_id_t in zip(boxes, boxes.id):
                track_id = int(track_id_t.item())
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                w = max(1.0, x2 - x1)
                h = max(1.0, y2 - y1)
                
                # Base status: Tracked Pellet
                label = f"ID:{track_id} Pellet"
                color = (255, 120, 0) # Cyan/Blue in BGR
                
                # Check if this pellet's identity has already been definitively locked
                if track_id in classified:
                    if classified[track_id] == "agglomerate":
                        label = f"ID:{track_id} Agglomerate"
                        color = (0, 0, 255) # Red in BGR
                    else:
                        label = f"ID:{track_id} Single"
                        color = (0, 255, 0) # Green in BGR
                else:
                    # Compute geometric elongation aspect ratio
                    aspect_ratio = max(w / h, h / w)
                    
                    # If elongated or unusually large, trigger the Suspicion Pipeline
                    if aspect_ratio >= suspicion_aspect_ratio:
                        if track_id not in suspicious_buffers:
                            suspicious_buffers[track_id] = []
                            
                        # Contextual square crop for the ResNet backbone
                        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
                        size = max(w, h) * 1.25
                        left = int(max(0, cx - size / 2.0))
                        right = int(min(frame_w, cx + size / 2.0))
                        top = int(max(0, cy - size / 2.0))
                        bottom = int(min(frame_h, cy + size / 2.0))
                        
                        crop = img_rgb[top:bottom, left:right]
                        if crop.size > 0 and (right - left) >= 4 and (bottom - top) >= 4:
                            crop_pil = Image.fromarray(crop).resize((CROP_SIZE, CROP_SIZE), Image.Resampling.BILINEAR)
                            crop_arr = np.array(crop_pil, dtype=np.float32).transpose(2, 0, 1) / 255.0
                            suspicious_buffers[track_id].append(crop_arr)
                            
                            buffer_len = len(suspicious_buffers[track_id])
                            label = f"ID:{track_id} Suspicious [{buffer_len}/{SEQUENCE_LENGTH}]"
                            color = (0, 165, 255) # Orange in BGR
                            
                            # Once SEQUENCE_LENGTH consecutive tumbling frames are captured, classify with AI
                            if buffer_len == SEQUENCE_LENGTH:
                                seq_tensor = torch.tensor(
                                    np.stack(suspicious_buffers[track_id]), 
                                    dtype=torch.float32
                                ).unsqueeze(0).to(device)
                                
                                with torch.no_grad():
                                    logits = seq_model(seq_tensor)
                                    probs = torch.softmax(logits, dim=1)
                                    pred = torch.argmax(probs, dim=1).item()
                                    confidence = probs[0, pred].item()
                                    
                                if pred == 1:
                                    classified[track_id] = "agglomerate"
                                    label = f"ID:{track_id} Agglomerate ({confidence*100:.0f}%)"
                                    color = (0, 0, 255)
                                else:
                                    classified[track_id] = "single"
                                    label = f"ID:{track_id} Single ({confidence*100:.0f}%)"
                                    color = (0, 255, 0)
                                    
                                # Release buffer memory; permanently locked!
                                del suspicious_buffers[track_id]
                                
                # Draw visually appealing styled bounding box & text
                cv2.rectangle(img_bgr, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
                cv2.rectangle(img_bgr, (int(x1), int(y1) - th - 6), (int(x1) + tw + 4, int(y1)), color, -1)
                cv2.putText(img_bgr, label, (int(x1) + 2, int(y1) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
                
        # Save output annotated frame
        out_path = os.path.join('output_frames', os.path.basename(frame_path))
        cv2.imwrite(out_path, img_bgr)
        
    print(f"Inference complete! {len(frames)} frames annotated and saved in output_frames/ folder.")

if __name__ == '__main__':
    infer()


## 🖼️ Step 6: Interactive Frame Inspector & Visualizer
Renders annotated detection frames directly inside the notebook with colored bounding boxes:
- 🔵 **Blue**: Regular Tracked Pellets
- 🟠 **Orange**: Suspicious Pellets (buffering 10-frame tumbling motion)
- 🟢 **Green**: Confirmed Single Pellets (verified by Temporal Attention)
- 🔴 **Red**: Confirmed Agglomerates (rigid multi-sphere tumble detected)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import cv2
import random
import glob

# Find all processed frames
output_images = sorted(glob.glob('output_frames/*.jpg'))

if output_images:
    # Randomly pick an annotated frame or specify an index (e.g. output_images[20])
    selected_path = random.choice(output_images)
    img_bgr = cv2.imread(selected_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(12, 12), dpi=120)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"Detection & Classification Output: {selected_path}", fontsize=15, pad=12)
    plt.tight_layout()
    plt.show()
else:
    print("No annotated images found in output_frames/. Please run Step 5 (infer.py) first!")
